# 🔐 Notebook 4: Secrets Management

Secrets are a **special kind of config** — things like DB passwords, API
keys, JWT signing keys, TLS private keys. Getting them wrong is how
companies end up on the front page of the news.

This notebook shows a clean **bad → good → best** for handling secrets
in a small Python service.


## 🛠️ Setup

```bash
cd 05-microservices/configuration-externalization
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 BAD: secrets in source code

Never do this. Once a secret is in git history it is effectively **public**
— rotating it is the only fix.


In [ ]:
# DO NOT DO THIS
DB_PASSWORD = "hunter2-super-secret"          # 😱 in git forever
STRIPE_KEY  = "sk_live_51HxxYYzzAAbbCCddEE"    # 😱

def connect():
    return f"postgres://app:{DB_PASSWORD}@db/app"

print("bad:", connect())


## 🟨 GOOD: load secrets from environment (or `.env` in dev)

The `.env` file stays **out of git** (`.gitignore` it). In production the
orchestrator (Kubernetes, ECS, systemd, ...) injects real secrets as
environment variables.


In [ ]:
import os, pathlib, tempfile

# Pretend this file lives at the project root and is gitignored.
dotenv = pathlib.Path(tempfile.gettempdir()) / ".env"
dotenv.write_text("DB_PASSWORD=dev-password\nSTRIPE_KEY=sk_test_123\n")

def load_dotenv(path: pathlib.Path) -> dict[str, str]:
    out = {}
    if not path.exists():
        return out
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        out[k.strip()] = v.strip()
    return out

# Real env always wins over .env so prod secrets aren't overridden by a file.
env = {**load_dotenv(dotenv), **os.environ}
print("loaded keys:", sorted(k for k in env if k in {"DB_PASSWORD", "STRIPE_KEY"}))


## 🙈 Never log secrets

The most common leak is not a hack — it's a stack trace or an INFO log
that accidentally prints the whole config object. Wrap secrets in a type
that **refuses** to render itself.


In [ ]:
class Secret:
    """String-like wrapper that never leaks its value via repr/str/logs."""
    __slots__ = ("_value",)

    def __init__(self, value: str):
        self._value = value

    def reveal(self) -> str:
        return self._value

    def __repr__(self) -> str:
        return "Secret('***')"
    __str__ = __repr__


db_password = Secret(env["DB_PASSWORD"])
print("log line :", f"connecting with password={db_password}")   # safe
print("actual   :", db_password.reveal()[:3] + "...")             # only when needed


## 🟩 BEST: a typed `Settings` with a `SecretStr`

Pydantic has `SecretStr` built-in — same idea as our `Secret` wrapper but
integrates cleanly with the `Settings` object from Notebook 1.


In [ ]:
from pydantic import BaseModel, SecretStr, Field

class AppSettings(BaseModel):
    db_host: str = "localhost"
    db_password: SecretStr
    stripe_key: SecretStr
    rate_limit: int = Field(default=10, ge=1)

settings = AppSettings(
    db_host="prod-db",
    db_password=env["DB_PASSWORD"],
    stripe_key=env["STRIPE_KEY"],
)

print("repr (safe to log):", settings)
print("masked pw         :", settings.db_password)
print("actual pw (only at use):", settings.db_password.get_secret_value())


## 📁 Secrets mounted as files (Docker / Kubernetes pattern)

Modern orchestrators often don't use env vars for secrets at all — they
mount each secret as a tiny file, e.g. `/run/secrets/db_password`. Benefits:

- The value doesn't appear in `/proc/<pid>/environ` or `docker inspect`.
- The orchestrator can **rotate** the file in place; the app can re-read it.
- File permissions (mode 0400) add another layer of protection.


In [ ]:
import pathlib, tempfile, os

# Simulate the /run/secrets/ directory you'd get in a container.
secrets_dir = pathlib.Path(tempfile.mkdtemp(prefix="run_secrets_"))
(secrets_dir / "db_password").write_text("rotated-v1")
(secrets_dir / "stripe_key").write_text("sk_test_rotated_v1")

def read_secret(name: str, mount: pathlib.Path = secrets_dir) -> str:
    # 1) prefer a mounted file (safer, rotatable)
    p = mount / name
    if p.exists():
        return p.read_text().strip()
    # 2) fall back to an env var (dev/local convenience)
    env_key = name.upper()
    if env_key in os.environ:
        return os.environ[env_key]
    raise KeyError(f"secret {name!r} not found in {mount} or env")

print("db_password starts with:", read_secret("db_password")[:8] + "...")


## 🔄 Rotation without restart

Secrets **will** be rotated — either on schedule or in response to an
incident. The app must pick up new values without a redeploy. One clean
way: read the secret through a function every time you need it (or cache
for a few seconds).


In [ ]:
import time

# Pretend the orchestrator rotates the file.
(secrets_dir / "db_password").write_text("rotated-v2")

print("after rotation  :", read_secret("db_password"))

# A tiny cached reader: re-read at most once per TTL.
class CachedSecret:
    def __init__(self, name: str, ttl_s: float = 30.0):
        self.name, self.ttl = name, ttl_s
        self._value, self._loaded_at = None, 0.0

    def get(self) -> str:
        now = time.time()
        if self._value is None or now - self._loaded_at > self.ttl:
            self._value = read_secret(self.name)
            self._loaded_at = now
        return self._value

pw = CachedSecret("db_password", ttl_s=0.0)  # ttl=0 -> always fresh, for demo
print("first read      :", pw.get())
(secrets_dir / "db_password").write_text("rotated-v3")
print("after another   :", pw.get())


### ⚠️ Accidental leaks at serialization time

`SecretStr` hides its value from `repr()` and plain `print()` — but be
careful when you dump the *whole* settings object:

- `model_dump()` → secrets are **masked** as `SecretStr('**********')` ✅
- `model_dump(mode="json")` / `model_dump_json()` in older pydantic could
  serialise the real value. Always **test** your logging and error-reporting
  paths with a fake secret like `"LEAKED-IF-YOU-SEE-ME"`.


In [ ]:
from pydantic import BaseModel, SecretStr

class Dump(BaseModel):
    api_key: SecretStr

d = Dump(api_key="LEAKED-IF-YOU-SEE-ME")
print("repr         :", d)
print("model_dump   :", d.model_dump())
print("dump json    :", d.model_dump_json())
# -> all should show ****** or similar, never the real value.


## 🏛️ Real secret stores (beyond env vars)

Environment variables are the lowest common denominator — fine for small
systems but they have drawbacks: visible in `/proc/<pid>/environ`, hard to
rotate, no audit log.

| Tool | What it gives you |
|---|---|
| **HashiCorp Vault** | Dynamic, short-lived secrets + audit log |
| **AWS Secrets Manager / SSM Parameter Store** | Managed KV with IAM + rotation |
| **GCP Secret Manager / Azure Key Vault** | Same, for their clouds |
| **Kubernetes Secrets** | Mounted as files/env in a pod (base64, *not* encrypted by default) |
| **SOPS / age** | Encrypt secrets *inside* git (keys stay in a KMS) |

The pattern is always the same: your **app reads an env var at startup**,
and something *outside* the app is responsible for putting the real value
there. Your code doesn't need to know which tool was used.


## ✅ Secrets checklist

- [ ] No secrets in source code or in the Docker image.
- [ ] `.env*` files are gitignored (and scanned for with tools like `gitleaks`).
- [ ] Secrets come from env vars or a secret manager at runtime.
- [ ] Secrets are wrapped (`SecretStr` / `Secret`) so they don't end up in logs.
- [ ] Secrets can be **rotated** without a code change.
- [ ] Access is audited (who read which secret, when).
